In [2]:
pip install pandas scikit-learn joblib numpy pyserial


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

data = pd.read_csv("crop_Recommendation_v2.csv")
print(data.head())



   Temperature   Humidity   Moisture    GasLevel  Crop
0    31.745401  89.014286  74.639879  479.597545  Rice
1    29.560186  73.119890  61.161672  559.852844  Rice
2    34.011150  84.161452  60.411690  590.972956  Rice
3    36.324426  74.246782  63.636499  355.021353  Rice
4    31.042422  80.495129  68.638900  387.368742  Rice


In [4]:
X = data[["Temperature", "Humidity", "Moisture", "GasLevel"]]
y = data["Crop"]


In [5]:
df = pd.read_csv("Crop_Recommendation_v2.csv")


In [6]:
import pandas as pd
import numpy as np

np.random.seed(42)

crops = {
    "Rice":      {"Temp": (28, 38), "Humidity": (70, 90), "Moisture": (60, 80), "Gas": (300, 600)},
    "Wheat":     {"Temp": (20, 30), "Humidity": (40, 60), "Moisture": (40, 60), "Gas": (150, 400)},
    "Maize":     {"Temp": (25, 35), "Humidity": (50, 70), "Moisture": (40, 70), "Gas": (200, 450)},
    "Sugarcane": {"Temp": (25, 40), "Humidity": (60, 85), "Moisture": (50, 80), "Gas": (400, 700)},
    "Cotton":    {"Temp": (22, 32), "Humidity": (50, 70), "Moisture": (30, 50), "Gas": (250, 500)},
    "Groundnut": {"Temp": (20, 30), "Humidity": (50, 60), "Moisture": (35, 55), "Gas": (150, 350)},
    "Millet":    {"Temp": (25, 38), "Humidity": (30, 50), "Moisture": (20, 40), "Gas": (100, 300)},
    "Barley":    {"Temp": (15, 25), "Humidity": (35, 55), "Moisture": (30, 50), "Gas": (100, 250)},
    "Soybean":   {"Temp": (22, 32), "Humidity": (60, 80), "Moisture": (50, 70), "Gas": (200, 400)},
    "Gram":      {"Temp": (18, 28), "Humidity": (40, 55), "Moisture": (25, 45), "Gas": (100, 250)},
}

data = []
samples_per_crop = 100  # 10 crops × 100 = 1000

for crop, ranges in crops.items():
    for _ in range(samples_per_crop):
        temp = np.random.uniform(*ranges["Temp"])
        hum = np.random.uniform(*ranges["Humidity"])
        mois = np.random.uniform(*ranges["Moisture"])
        gas = np.random.uniform(*ranges["Gas"])
        data.append([temp, hum, mois, gas, crop])

df = pd.DataFrame(data, columns=["Temperature", "Humidity", "Moisture", "GasLevel", "Crop"])
df.to_csv("Crop_Recommendation_v2.csv", index=False)
print("✅ Dataset created: Crop_Recommendation_v2.csv")


✅ Dataset created: Crop_Recommendation_v2.csv


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("Crop_Recommendation_v2.csv")

# Encode labels
le = LabelEncoder()
df['Crop'] = le.fit_transform(df['Crop'])

# Split data
X = df[['Temperature', 'Humidity', 'Moisture', 'GasLevel']]
y = df['Crop']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

# Evaluate
pred = model.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, pred) * 100)
print(classification_report(y_test, pred, target_names=le.classes_))


KeyboardInterrupt: 

In [ ]:
import joblib
joblib.dump(model, "crop_model.pkl")
joblib.dump(le, "label_encoder.pkl")


['label_encoder.pkl']

In [ ]:
import serial
import time

# Arduino se connect kar
ser = serial.Serial('COM5', 9600, timeout=2)  
time.sleep(2)  # wait for Arduino to reset

print("Collecting sensor data...\n")

for i in range(10):  # 10 readings test ke liye
    if ser.in_waiting > 0:  # wait until data available
        data = ser.readline().decode('utf-8', errors='ignore').strip()
        print(data)
    time.sleep(1)  

ser.close()
print("\nDone!")



33.80,33.10,939,197
33.80,33.00,938,198
33.80,33.00,938,199
33.80,32.90,937,198

Done!


In [9]:
import serial
import time
import pandas as pd

# COM port Arduino ka (check Arduino IDE → Tools → Port)
ser = serial.Serial('COM5', 9600, timeout=2)
time.sleep(3)

data_list = []
print("Collecting sensor data...\n")

for i in range(10):  # 10 readings le lo (test ke liye)
    line = ser.readline().decode('utf-8', errors='ignore').strip()
    if line:
        print(line)
        values = line.split(',')
        if len(values) == 4:
            data_list.append(values)
    time.sleep(1)

ser.close()

# Save readings to CSV
df = pd.DataFrame(data_list, columns=['Temperature','Humidity','SoilMoisture','GasLevel'])
df.to_csv('sensor_data.csv', index=False)
print("\n✅ Data saved to sensor_data.csv")



23.50,83.70,883,14
23.50,83.80,882,15
23.50,83.70,884,13
23.50,83.70,886,14
23.50,83.70,896,13
23.50,83.70,900,13
23.60,83.60,899,14
23.50,83.70,902,14
23.60,83.60,901,13
23.60,83.60,903,14

✅ Data saved to sensor_data.csv


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Dataset load
data = pd.read_csv('Crop_Recommendation_v2.csv')

X = data[['Temperature','Humidity','Moisture','GasLevel']]
y = data['Crop']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("✅ Model Accuracy:", accuracy_score(y_test, y_pred)*100, "%")


✅ Model Accuracy: 81.0 %


In [12]:
# Read the CSV saved from Arduino
sensor_data = pd.read_csv('sensor_data.csv')

# Take the last/latest reading
latest = sensor_data.iloc[-1].values.reshape(1, -1)

prediction = model.predict(latest)
print("\n🌱 Recommended Crop:", prediction[0])




🌱 Recommended Crop: Rice


c:\Users\yadav\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd

df = pd.read_csv("Crop_Recommendation_v2.csv")
print(df['Crop'].value_counts())


Crop
Rice         100
Wheat        100
Maize        100
Sugarcane    100
Cotton       100
Groundnut    100
Millet       100
Barley       100
Soybean      100
Gram         100
Name: count, dtype: int64


In [ ]:
# Read the CSV saved from Arduino
sensor_data = pd.read_csv('sensor_data.csv')

# Take the last/latest reading
latest = sensor_data.iloc[-1].values.reshape(1, -1)

prediction = model.predict(latest)
print("\n🌱 Recommended Crop:", prediction[0])


🌱 Recommended Crop: Rice


c:\Users\yadav\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
